# MINE4201 - SR - Taller 1 - Funciones para producción

**Grupo 3**

Lina María Ojeda Amaya Código: 202112324

Diego Felipe Tolosa Código: 201413235

Juan Manuel Rivera López Código: 201534131

Johana Alejandra Rátiva Mora Código: 202513844

# Importación de librerías y datos

In [76]:
import numpy as np
import os
import pandas as pd
import random
from matplotlib import pyplot as plt
import time
from collections import defaultdict


from scipy.sparse import csr_matrix

from sklearn.preprocessing import normalize
from tqdm import tqdm

#Para garantizar reproducibilidad en resultados
seed = 10
random.seed(seed)
np.random.seed(seed)

In [77]:
base_path = "C:/Users/jmriv/OneDrive - Universidad de los andes/Semestres uniandes/2026-1/Sistemas de recomendación/Talleres"
links_path = os.path.join(base_path, "link.csv")
movies_path = os.path.join(base_path, "movie.csv")
ratings_path = os.path.join(base_path, "rating.csv")

In [111]:
ratings = pd.read_csv(ratings_path)
ratings.head()

,userId,movieId,rating,timestamp
0,1,2,3.5,2005-04-02 23:53:47
1,1,29,3.5,2005-04-02 23:31:16
2,1,32,3.5,2005-04-02 23:33:39
3,1,47,3.5,2005-04-02 23:32:07
4,1,50,3.5,2005-04-02 23:29:40


In [79]:
movies = pd.read_csv(movies_path)
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [80]:
top_movies = pd.read_csv("Top_popular_movies.csv", index_col=0)
top_movies.head(10)

,movieId,mean,count,var,title,weighted_mean
315,318,4.446990,63366,0.514842,"Shawshank Redemption, The (1994)",4.446947
843,858,4.364732,41355,0.705393,"Godfather, The (1972)",4.364671
49,50,4.334372,47006,0.572721,"Usual Suspects, The (1995)",4.334321
523,527,4.310175,50054,0.681320,Schindler's List (1993),4.310128
23694,113315,4.500000,11,0.250000,Zero Motivation (Efes beyahasei enosh) (2014),4.291185
1195,1221,4.275641,27398,0.747358,"Godfather: Part II, The (1974)",4.275558
1935,2019,4.274180,11611,0.701326,Seven Samurai (Shichinin no samurai) (1954),4.273986
887,904,4.271334,17449,0.566462,Rear Window (1954),4.271205
7356,7502,4.263182,4305,0.820357,Band of Brothers (2001),4.262669
895,912,4.258327,24349,0.745426,Casablanca (1942),4.258237


In [81]:
top_movies[:10]['movieId'].values

array([   318,    858,     50,    527, 113315,   1221,   2019,    904,
         7502,    912], dtype=int64)

# Construcción de la matriz de utilidad

In [82]:
test = ratings.groupby("userId", group_keys=False).sample(frac=0.2, random_state=42)
train = ratings.drop(test.index)

In [83]:
user_cat = train.userId.astype("category")
item_cat = train.movieId.astype("category")

user_ids = user_cat.cat.codes
item_ids = item_cat.cat.codes

R_train = csr_matrix(
    (train.rating, (user_ids, item_ids))
)

In [84]:
user_map = dict(zip(user_cat.cat.categories, range(len(user_cat.cat.categories))))
item_map = dict(zip(item_cat.cat.categories, range(len(item_cat.cat.categories))))
movie_reverse_map = {idx: movieId for movieId, idx in item_map.items()}

In [85]:
R_train.data = R_train.data.astype(np.float32)

# Predicciones usuario-usuario

In [86]:
best_sim = 'pearson'
best_K = 100
threshold = 0.5

In [87]:
N = R_train.shape[0]

counts = np.diff(R_train.indptr)
sums = np.add.reduceat(R_train.data, R_train.indptr[:-1])
user_means = np.divide(sums, counts, out=np.zeros_like(sums), where=counts!=0)

In [88]:
with np.load(f"Train_neighbours/users_{best_sim}_top100_neighbors.npz") as data:
    top_k_indices = data["indices"]
    top_k_sims = data["sims"]
print("min sim:", top_k_sims.min())
print("max sim:", top_k_sims.max())

min sim: -1.0
max sim: 0.9672295


In [90]:
def get_user_neighbors(user_id, K):
    with np.load(f"Train_neighbours/users_{best_sim}_top100_neighbors.npz") as data:
        top_k_indices = data["indices"]
        top_k_sims = data["sims"]
    print("min sim:", top_k_sims.min())
    print("max sim:", top_k_sims.max())
    return top_k_indices[user_id][:K], top_k_sims[user_id][:K]

In [91]:
def recommend_movies(user_id, K=100, threshold=0.5, N=10):

    if user_id not in user_map:
        return top_movies[:N]['movieId'].values
    user_idx = user_map[user_id]

    neighbors, similarities = get_user_neighbors(user_idx, K)
    if len(neighbors) == 0:
        return top_movies[:N]['movieId'].values
    
    numerators = defaultdict(float)
    denominators = defaultdict(float)

    for neighbor, sim in zip(neighbors, similarities):
        mask = similarities >= threshold
        neighbors = neighbors[mask]
        similarities = similarities[mask]


        start = R_train.indptr[neighbor]
        end = R_train.indptr[neighbor+1]

        movies = R_train.indices[start:end]
        ratings = R_train.data[start:end]

        centered = ratings - user_means[neighbor]

        for m, r in zip(movies, centered):
            numerators[m] += sim * r
            denominators[m] += abs(sim)

    predictions = {}

    for m in numerators:

        if denominators[m] == 0:
            continue

        pred = user_means[user_idx] + numerators[m] / denominators[m]
        predictions[m] = pred

    # remove seen movies
    start = R_train.indptr[user_idx]
    end = R_train.indptr[user_idx + 1]
    seen = set(R_train.indices[start:end])

    candidates = {movie_reverse_map[m]: s for m, s in predictions.items() if m not in seen}

    top_movies = sorted(candidates.items(), key=lambda x: x[1], reverse=True)[:N]

    print(top_movies)

    return [movieId for movieId, _ in top_movies]

In [92]:
recommend_movies(101, best_K, threshold)

min sim: -1.0
max sim: 0.9672295
[(73, 5.244897767913751), (745, 5.234693795928724), (800, 5.21395653769032), (1036, 5.210692104655427), (347, 5.1604082431097025), (101, 5.0903428710619005), (1089, 5.0348589058355575), (1032, 5.0204081535339355), (123, 5.00286444135895), (709, 4.77512506880944)]


[73, 745, 800, 1036, 347, 101, 1089, 1032, 123, 709]

In [93]:
top_movies[top_movies['movieId'] == 73]

,movieId,mean,count,var,title,weighted_mean
72,73,3.806768,2955,1.081132,"Misérables, Les (1995)",3.806483


# Predicciones item-item

In [ ]:
best_sim = 'pearson'
best_K = 100
threshold = 0.5

In [ ]:
R_items = R_train.T

N = R_items.shape[0]

counts = np.diff(R_items.indptr)
sums = np.add.reduceat(R_items.data, R_items.indptr[:-1])
item_means = np.divide(sums, counts, out=np.zeros_like(sums), where=counts!=0)

In [ ]:
with np.load(f"Train_neighbours/items_{best_sim}_top100_neighbors.npz") as data:
    top_k_indices = data["indices"]
    top_k_sims = data["sims"]
print("min sim:", top_k_sims.min())
print("max sim:", top_k_sims.max())

In [ ]:
def get_item_neighbors(item_id, K):
    with np.load(f"Train_neighbours/items_{best_sim}_top100_neighbors.npz") as data:
        top_k_indices = data["indices"]
        top_k_sims = data["sims"]
    print("min sim:", top_k_sims.min())
    print("max sim:", top_k_sims.max())
    return top_k_indices[item_id][:K], top_k_sims[item_id][:K]

In [ ]:
def recommend_movies_for_item(item_id, K=100, threshold=0.5, N=10):

    if item_id not in item_map:
        return top_movies[:N]['movieId'].values
    item_idx = item_map[item_id]

    neighbors, similarities = get_item_neighbors(item_idx, K)
    if len(neighbors) == 0:
        print("Top popular")
        return top_movies[:N]['movieId'].values
    
    numerators = defaultdict(float)
    denominators = defaultdict(float)

    for neighbor, sim in zip(neighbors, similarities):
        mask = similarities >= threshold
        neighbors = neighbors[mask]
        similarities = similarities[mask]


        start = R_items.indptr[neighbor]
        end = R_items.indptr[neighbor+1]

        users = R_items.indices[start:end]
        ratings = R_items.data[start:end]

        centered = ratings - item_means[users]

        for m, r in zip(users, centered):
            numerators[m] += sim * r
            denominators[m] += abs(sim)

    predictions = {}

    for m in numerators:

        if denominators[m] == 0:
            continue

        pred = item_means[item_idx] + numerators[m] / denominators[m]
        predictions[m] = pred

    # remove seen movies
    start = R_items.indptr[item_idx]
    end = R_items.indptr[item_idx + 1]

    candidates = {movie_reverse_map[m]: s for m, s in predictions.items() if m not in seen}

    top_movies = sorted(candidates.items(), key=lambda x: x[1], reverse=True)[:N]

    print(top_movies)

    return [movieId for movieId, _ in top_movies]

# Manejo de nuevos usuarios y actualizar items

## Nuevos usuarios

In [133]:
new_rows = []

for movie in train.movieId.unique()[:10]:
    new_rows.append({
        "userId": 138494,
        "movieId": movie,
        "rating": random.randint(1,10) * 0.5,
        "timestamp": None
    })

train = pd.concat([train, pd.DataFrame(new_rows)], ignore_index=True)

In [134]:
train.tail()

,userId,movieId,rating,timestamp
16001032,138494,112,1.0,None
16001033,138494,151,1.5,None
16001034,138494,223,3.5,None
16001035,138494,253,5.0,None
16001036,138494,293,3.0,None


In [145]:
user_cat = train.userId.astype("category")
item_cat = train.movieId.astype("category")

user_ids = user_cat.cat.codes
item_ids = item_cat.cat.codes

R_train = csr_matrix(
    (train.rating, (user_ids, item_ids))
)

user_map = dict(zip(user_cat.cat.categories, range(len(user_cat.cat.categories))))
item_map = dict(zip(item_cat.cat.categories, range(len(item_cat.cat.categories))))
movie_reverse_map = {idx: movieId for movieId, idx in item_map.items()}

In [144]:
def recalculate_neighbours(R, userIdx, K, metric, neighbours, similarities):
    
    neighbours = np.vstack([neighbours, np.zeros((1, K), dtype=np.int32)])
    similarities = np.vstack([similarities, np.zeros((1, K), dtype=np.float32)])

    # Pre-process matrices based on the metric
    t0 = time.time()
    if metric == "cosine":
        R_processed = normalize(R, axis=1)
    elif metric == "pearson":
        R_processed = R.copy()
        counts = np.diff(R_processed.indptr)
        sums = np.add.reduceat(R_processed.data, R_processed.indptr[:-1])
        means = sums / counts
        R_processed.data -= np.repeat(means, counts)
        R_processed = normalize(R_processed, axis=1)

    elif metric == "jaccard":
        B = R.copy()
        B.data = np.ones_like(B.data) # Binary matrix for Jaccard
        row_sums = np.array(B.sum(axis=1)).flatten() # Count of non-zero ítems in the row
    print("Normalization:", time.time() - t0)

    t0 = time.time()
    start_idx = userIdx
    end_idx = start_idx + 1
        
    # 1. Calculate similarity for the current user
    if metric in ["cosine", "pearson"]:
        # Returns a sparse matrix of size (batch_size, N)
        batch_sim_sparse = R_processed[start_idx:end_idx] @ R_processed.T
        
    elif metric == "jaccard":
        B_batch = B[start_idx:end_idx]
        batch_sim_sparse = B_batch @ B.T # Calculate the union

    for i in range(batch_sim_sparse.shape[0]):

        row = batch_sim_sparse.getrow(i)

        sims = row.data
        indices = row.indices

        u = start_idx + i
        mask = indices != u
        indices = indices[mask]
        sims = sims[mask]

        if len(sims) == 0:
            neighbours[start_idx+i, 0] = -1
            similarities[start_idx+i, 0] = -1
            continue

        if metric == 'jaccard':
            unions = row_sums[u] + row_sums[indices] - sims
            sims = sims / unions

        k = min(K, len(sims))

        top_idx = np.argpartition(-sims, k-1)[:k] # Finds the top K neighbour's positions for the user

        neighbours[start_idx+i, :k] = indices[top_idx]
        similarities[start_idx+i, :k] = sims[top_idx]
    print("Total time:", time.time() - t0)

    return neighbours, similarities

In [152]:
with np.load(f"Train_neighbours/users_{best_sim}_top100_neighbors.npz") as data:
    top_k_indices = data["indices"]
    top_k_sims = data["sims"]
print("min sim:", top_k_sims.min())
print("max sim:", top_k_sims.max())

min sim: -1.0
max sim: 0.9672295


In [154]:
len(top_k_indices)

138493

In [155]:
top_k_indices, top_k_sims = recalculate_neighbours(
    R_train,
    user_map[138494],
    100,
    'pearson',
    top_k_indices,
    top_k_sims)
len(top_k_indices)

Normalization: 1.0224354267120361
Total time: 1.6050753593444824


138494

## Actualizar un usuario

In [157]:
def update_neighbours(R, userIdx, K, metric, neighbours, similarities):

    # Pre-process matrices based on the metric
    t0 = time.time()
    if metric == "cosine":
        R_processed = normalize(R, axis=1)
    elif metric == "pearson":
        R_processed = R.copy()
        counts = np.diff(R_processed.indptr)
        sums = np.add.reduceat(R_processed.data, R_processed.indptr[:-1])
        means = sums / counts
        R_processed.data -= np.repeat(means, counts)
        R_processed = normalize(R_processed, axis=1)

    elif metric == "jaccard":
        B = R.copy()
        B.data = np.ones_like(B.data) # Binary matrix for Jaccard
        row_sums = np.array(B.sum(axis=1)).flatten() # Count of non-zero ítems in the row
    print("Normalization:", time.time() - t0)

    t0 = time.time()
    start_idx = userIdx
    end_idx = start_idx + 1
        
    # 1. Calculate similarity for the current user
    if metric in ["cosine", "pearson"]:
        # Returns a sparse matrix of size (batch_size, N)
        batch_sim_sparse = R_processed[start_idx:end_idx] @ R_processed.T
        
    elif metric == "jaccard":
        B_batch = B[start_idx:end_idx]
        batch_sim_sparse = B_batch @ B.T # Calculate the union

    for i in range(batch_sim_sparse.shape[0]):

        row = batch_sim_sparse.getrow(i)

        sims = row.data
        indices = row.indices

        u = start_idx + i
        mask = indices != u
        indices = indices[mask]
        sims = sims[mask]

        if len(sims) == 0:
            neighbours[start_idx+i, 0] = -1
            similarities[start_idx+i, 0] = -1
            continue

        if metric == 'jaccard':
            unions = row_sums[u] + row_sums[indices] - sims
            sims = sims / unions

        k = min(K, len(sims))

        top_idx = np.argpartition(-sims, k-1)[:k] # Finds the top K neighbour's positions for the user

        neighbours[start_idx+i, :k] = indices[top_idx]
        similarities[start_idx+i, :k] = sims[top_idx]
    print("Total time:", time.time() - t0)

    return neighbours, similarities